In [ ]:
# Libraries Importing

# Basic Data Handling
import pandas as pd
import numpy as np

# Regular Expressions (for text cleaning)
import re

# Natural Language Processing
import nltk
from nltk.corpus import stopwords

# Machine Learning Models
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression

# Model Evaluation
from sklearn.metrics import classification_report, confusion_matrix

# Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
# 1) Data acquisition
import pandas as pd
data=pd.read_csv('spam_Emails_data.csv')
print(data.head())
print(data.columns)
print(data.isnull().sum())
print(data['label'].value_counts())

  label                                               text
0  Spam  viiiiiiagraaaa\nonly for the ones that want to...
1   Ham  got ice thought look az original message ice o...
2  Spam  yo ur wom an ne eds an escapenumber in ch ma n...
3  Spam  start increasing your odds of success & live s...
4   Ham  author jra date escapenumber escapenumber esca...
Index(['label', 'text'], dtype='object')
label    0
text     0
dtype: int64
label
Ham     1761
Spam    1577
Name: count, dtype: int64


In [ ]:
# 2)  Cleaning the data (full cleaning)
import re
import pandas as pd
import numpy as np

def clean_text(text):
    # Check if the value is NaN or not a string
    if isinstance(text, float) and np.isnan(text):
        return ""

    # Convert to string if it's not already
    text = str(text)

    # Lowercase
    text = text.lower()
    # Remove URLs
    text = re.sub(r'http\S+|www.\S+', '', text)
    # Remove punctuations and numbers
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'http\S+|www.\S+', ' <LINK> ', text)
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply the function and handle NaN values
data['clean_text'] = data['text'].apply(clean_text)

In [ ]:
# Removing stop keywords
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    return ' '.join([word for word in text.split() if word not in stop_words])

data['clean_text'] = data['clean_text'].apply(remove_stopwords)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
# 3) Text feature extraction TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)  # Take top 5000 most important words
X_text = tfidf.fit_transform(data['clean_text']).toarray()

In [ ]:

#  now extract structural patterns like: these are features extracted

# 1) Email Length (more words = might be spam)
# Convert non-string entries to string or empty string
data['email_length'] = data['text'].fillna('').astype(str).apply(len)

# 2) Number of Links (spams often have lots of links)
import re

def count_links(text):
    if isinstance(text, str):
        return len(re.findall(r'http\S+', text))
    return 0

data['num_links'] = data['text'].apply(count_links)


# 3) Capital Letter Ratio (spams often SHOUT)
def capital_ratio(text):
    if not isinstance(text, str):
        text = str(text) if text is not None else ''
    capitals = sum(1 for c in text if c.isupper())
    return capitals / max(1, len(text))

data['capital_ratio'] = data['text'].apply(capital_ratio)

# 4) Presence of Spammy Words ("FREE", "WIN", "URGENT", etc.)
def has_spammy_words(text):
    spammy = ['free', 'win', 'guarantee', 'winner', 'urgent']
    text = str(text).lower()
    return int(any(word in text for word in spammy))

data['spammy_words'] = data['text'].apply(has_spammy_words)



In [ ]:
# Now Merging all 4 features
import numpy as np

# Ensure no NaNs in structural features : safe handling data
data[['email_length', 'num_links', 'capital_ratio', 'spammy_words']] = \
    data[['email_length', 'num_links', 'capital_ratio', 'spammy_words']].fillna(0)

# Extract structural features as a NumPy array
X_structural = data[['email_length', 'num_links', 'capital_ratio', 'spammy_words']].values

# Merge TF-IDF text features with structural features
X_final = np.hstack((X_text, X_structural))   # complete feature set for machine learning features + all data

In [ ]:
# prepare labels
y_final = data['label'].map({'ham': 0, 'spam': 1})  # Map ham -> 0, spam -> 1

In [ ]:
#print(y_final)
print(data['label'].isna().sum())  # This will show the number of NaN values in your labels column

0


In [ ]:
# print(X_final.shape)
# print(data['label'].shape)  # Check the shape of the label column
# Remove rows where 'label' column has NaN values

# Remove rows where 'label' is NaN
data = data.dropna(subset=['label'])

# Reset the index to keep things clean (optional but recommended)
data = data.reset_index(drop=True)

# Now extract labels
y_final = data['label'].values

In [ ]:
# check if any NaNs remain
print(data['label'].isnull().sum())  # Correct way for object/string types

0


In [ ]:
# check shape of X and y final
print("X_final shape:", X_final.shape)
print("y_final shape:", y_final.shape)

X_final shape: (3338, 5004)
y_final shape: (3338,)


In [ ]:
# cheching features of first mail
print("First email features (X_final):", X_final[0])

First email features (X_final): [0. 0. 0. ... 0. 0. 0.]


In [ ]:
print(data['label'].unique())
data['label'] = data['label'].str.strip().str.lower()
y_final = data['label'].map({'ham': 0, 'spam': 1})

['Spam' 'Ham']


In [ ]:
# check label values
print("First 10 labels:", y_final[:10].values)

First 10 labels: [1 0 1 1 0 1 0 0 0 0]


In [ ]:
print(y_final.value_counts())

label
0    1761
1    1577
Name: count, dtype: int64


In [ ]:
print(X_final[10])

[0. 0. 0. ... 0. 0. 1.]


In [ ]:
print(data['clean_text'])

0       viiiiiiagraaaa ones want make scream prodigy s...
1       got ice thought look az original message ice o...
2       yo ur wom ne eds escapenumber ch n b e th n f ...
3       start increasing odds success live sexually he...
4       author jra date escapenumber escapenumber esca...
                              ...                        
3333    original message jamey estes jason dobbs cc oo...
3334    srea takes investors second climb escapenumber...
3335               escapelong dynpzfgtf x kkxgxz xhwro wq
3336    greetings amazon com weve recently learned sup...
3337    bodylabel subject great parttime summer job di...
Name: clean_text, Length: 3338, dtype: object


In [ ]:
# saving cleaned data after data preprocessing
data.to_csv('cleaned_dataset.csv', index=False)

In [ ]:
# Testing training Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_final, y_final, test_size=0.2, random_state=42)
# 80% training 20% testing

In [ ]:
# 4) Now Training ML Models

# 1) Naive Bayes
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

MultinomialNB()

In [ ]:
# 2) Logistic Regression
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [ ]:
# 3)  Random Forest
from sklearn.ensemble import RandomForestClassifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)


RandomForestClassifier(random_state=42)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# List of models to compare
models = [nb_model, lr_model, rf_model]
names = ['Naive Bayes', 'Logistic Regression', 'Random Forest']

# Track best model
best_model = None
best_f1 = 0
best_name = ""
best_y_pred = None

print("Comparing Models...\n")

# Evaluate each model
for model, name in zip(models, names):
    y_pred = model.predict(X_test)

    report = classification_report(y_test, y_pred, output_dict=True)
    f1_spam = report['1']['f1-score']

    print(f"  {name}")
    print(f"F1-Score (Spam): {f1_spam:.4f}")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print()

    # Update best model based on spam F1-score
    if f1_spam > best_f1:
        best_f1 = f1_spam
        best_model = model
        best_name = name
        best_y_pred = y_pred

# Final best model results
print("Final Results for Best Model:", best_name)
print(f"Accuracy: {accuracy_score(y_test, best_y_pred):.4f}")
print("Classification Report:")
print(classification_report(y_test, best_y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, best_y_pred))


Comparing Models...

  Naive Bayes
F1-Score (Spam): 0.8709
Accuracy: 0.8832
Confusion Matrix:
[[327  22]
 [ 56 263]]

  Logistic Regression
F1-Score (Spam): 0.9484
Accuracy: 0.9506
Confusion Matrix:
[[332  17]
 [ 16 303]]

  Random Forest
F1-Score (Spam): 0.9323
Accuracy: 0.9356
Confusion Matrix:
[[329  20]
 [ 23 296]]

Final Results for Best Model: Logistic Regression
Accuracy: 0.9506
Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.95      0.95       349
           1       0.95      0.95      0.95       319

    accuracy                           0.95       668
   macro avg       0.95      0.95      0.95       668
weighted avg       0.95      0.95      0.95       668

Confusion Matrix:
[[332  17]
 [ 16 303]]


In [ ]:
import joblib

# Save trained model
joblib.dump(lr_model, "spam_classifier.pkl")

# Also save TF-IDF vectorizer if needed
joblib.dump(tfidf, "tfidf_vectorizer.pkl")

['tfidf_vectorizer.pkl']

In [ ]:
import joblib
model = joblib.load("spam_classifier.pkl")  # Your saved Logistic Regression

In [ ]:
# Now deploy Best Model to Gradio
!pip install gradio --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.2/54.2 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.3/323.3 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 97.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 4.0 MB/s eta 0:00:00


In [ ]:
import gradio as gr
import joblib
import numpy as np
import matplotlib.pyplot as plt
import re

# Load best model (Logistic Regression) and vectorizer
model = joblib.load("spam_classifier.pkl")
vectorizer = joblib.load("tfidf_vectorizer.pkl")

# Global counters
email_count = 0
spam_count = 0
ham_count = 0

# === Preprocessing functions ===

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www.\S+", "", text)  # remove links
    text = re.sub(r"[^a-z\s]", "", text)         # remove punctuation/numbers
    text = re.sub(r"\s+", " ", text).strip()     # remove extra spaces
    return text

def has_spammy_words(text):
    spam_words = ["free", "win", "guarantee", "urgent", "winner", "click", "offer", "money"]
    return int(any(word in text.lower() for word in spam_words))

def count_links(text):
    if isinstance(text, str):
        return len(re.findall(r'http\S+', text))
    return 0


def capital_ratio(text):
    caps = sum(1 for c in text if c.isupper())
    return caps / max(1, len(str(text)))

def get_structural_features(text):
    return np.array([
        len(text),
        count_links(text),
        capital_ratio(text),
        has_spammy_words(text)
    ]).reshape(1, -1)

# === Main prediction function ===

def predict_email(email):
    global email_count, spam_count, ham_count

    email_count += 1

    cleaned = clean_text(email)
    tfidf_vector = vectorizer.transform([cleaned]).toarray()
    struct_features = get_structural_features(email)
    final_input = np.hstack((tfidf_vector, struct_features))

    pred = model.predict(final_input)[0]
    proba = model.predict_proba(final_input)[0][pred]

    label = "SPAM" if pred == 1 else "HAM"
    confidence = f"{proba * 100:.2f}%"

    if pred == 1:
        spam_count += 1
    else:
        ham_count += 1

    # Create live bar chart
    fig, ax = plt.subplots()
    ax.bar(["Spam", "Ham"], [spam_count, ham_count], color=["red", "green"])
    ax.set_title("Spam vs Ham Email Count")
    ax.set_ylabel("Emails")
    plt.tight_layout()

    return label, confidence, email_count, fig


In [ ]:

# === Gradio UI ===

interface = gr.Interface(
    fn=predict_email,
    inputs=gr.Textbox(label="📥 Enter Email Text"),
    outputs=[
        gr.Textbox(label="Prediction (SPAM / HAM)"),
        gr.Textbox(label="Confidence"),
        gr.Number(label="Total Emails Tested"),
        gr.Plot(label="Spam vs Ham Count")
    ],
    title="📧 Intelligent Email Spam Detection Using Machine Learning and Natural Language Processing",
    description="Trained using Logistic Regression with TF-IDF and structural features. Enter an email to classify it as spam or not."
)
interface.launch()


It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4774deaf91cf27a5aa.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
